# Embeddings + preparación de texto (core Capítulo 2)

Este notebook recrea el **código central** del Capítulo 2 (LLMs-from-scratch) y agrega **explicaciones propias** + un experimento pequeño con `max_length` y `stride`.

## Por qué importa la tokenización (LLMs + sistemas agénticos)

Los LLMs no pueden consumir strings crudos directamente; operan sobre **enteros (IDs de tokens)**. La tokenización es la interfaz que convierte texto en una secuencia discreta que el modelo puede aprender.

En **sistemas agénticos**, la tokenización también afecta costo y comportamiento: prompts de herramientas, salidas JSON y contextos largos pueden tokenizarse muy distinto. Un cambio pequeño de formato puede cambiar la cantidad de tokens (y por tanto latencia/costo) y también qué cabe dentro de la ventana de contexto.

In [ ]:
import sys
import subprocess

def ensure_package(pkg: str) -> None:
    """Instala el paquete si no está disponible en el entorno actual."""
    try:
        __import__(pkg)
        return
    except Exception:
        pass
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

# Instalar solo si falta
ensure_package('requests')
ensure_package('tiktoken')
ensure_package('torch')

In [ ]:
from importlib.metadata import version

print('python:', sys.version.split()[0])
print('torch versión:', version('torch'))
print('tiktoken versión:', version('tiktoken'))

## Cargar texto crudo
Usaremos *The Verdict* (dominio público) como un corpus pequeño. Es intencionalmente corto, pero alcanza para demostrar el pipeline: **texto → tokens → IDs → muestras de entrenamiento → embeddings**.

In [ ]:
import os
import requests

txt_path = 'the-verdict.txt'
if not os.path.exists(txt_path):
    url = ('https://raw.githubusercontent.com/rasbt/'
           'LLMs-from-scratch/main/ch02/01_main-chapter-code/'
           'the-verdict.txt')
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    with open(txt_path, 'wb') as f:
        f.write(resp.content)

with open(txt_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

print('Número total de caracteres:', len(raw_text))
print(raw_text[:120])

## Tokenizador simple (basado en regex)

Antes de usar tokenizadores de producción (BPE), conviene ver qué *está haciendo* la tokenización.
Este tokenizador separa por espacios en blanco y puntuación. No es como tokeniza GPT-2, pero da intuición de por qué necesitamos IDs y un vocabulario.

In [ ]:
import re

text = 'Hello, world. Is this-- a test?'
tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
tokens = [t.strip() for t in tokens if t.strip()]
print(tokens)

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print('Primeros 30 tokens:', preprocessed[:30])
print('Cantidad de tokens:', len(preprocessed))

## ¿Por qué construir un vocabulario + IDs de tokens?

Las redes neuronales operan con números, así que mapeamos cada token a un ID entero. Esto crea un **espacio discreto de símbolos** (IDs) que luego proyectamos a un **espacio vectorial continuo** (embeddings).

En la práctica, los vocabularios buscan balancear: (1) cobertura (pocos desconocidos), (2) compacidad (tablas de embeddings/softmax más chicas) y (3) eficiencia (encode/decode rápido).

In [ ]:
all_words = sorted(set(preprocessed))
vocab = {token: integer for integer, token in enumerate(all_words)}
print('Tamaño del vocabulario:', len(vocab))

# Mostrar algunas entradas
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 20:
        break

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        text = ' '.join([self.int_to_str[i] for i in ids])
        return re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)


tokenizer_v1 = SimpleTokenizerV1(vocab)
sample = '"It\'s the last he painted, you know," Mrs. Gisburn said.'
ids = tokenizer_v1.encode(sample)
print('Codificado:', ids[:20], '...')
print('Decodificado:', tokenizer_v1.decode(ids))

# Demostrar fallo con token desconocido
try:
    tokenizer_v1.encode('Hello, do you like tea?')
except KeyError as e:
    print('V1 falla con token desconocido:', e)

## Por qué importan los tokens especiales (control de contexto)

Los tokens especiales nos permiten representar **límites de secuencia** y **comportamiento ante desconocidos/padding** de forma consistente.

En flujos agénticos, delimitadores explícitos (como fin-de-texto o separadores de herramientas) marcan la diferencia entre: *un flujo coherente de instrucciones* vs. *múltiples chunks independientes* — lo cual cambia fuerte a qué atiende el modelo y cómo decide actuar.

In [ ]:
# Extender vocabulario con tokens especiales
all_tokens = sorted(set(preprocessed))
all_tokens.extend(['<|endoftext|>', '<|unk|>'])
vocab2 = {token: integer for integer, token in enumerate(all_tokens)}

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int else '<|unk|>' for item in preprocessed]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        text = ' '.join([self.int_to_str[i] for i in ids])
        return re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)

tokenizer_v2 = SimpleTokenizerV2(vocab2)
text1 = 'Hello, do you like tea?'
text2 = 'In the sunlit terraces of the palace.'
joined = ' <|endoftext|> '.join((text1, text2))
print('Texto unido:', joined)
print('Codificado:', tokenizer_v2.encode(joined))
print('Decodificado:', tokenizer_v2.decode(tokenizer_v2.encode(joined)))

## Por qué BPE (tiktoken) importa en LLMs reales

Los tokenizadores por palabras/reglas se rompen con palabras desconocidas. Los LLMs modernos (como GPT-2) usan **tokenización subword** (BPE) para representar palabras raras, nombres y typos como combinaciones de unidades más pequeñas.

Esto mejora la cobertura sin explotar el tamaño del vocabulario: obtenés una tabla de embeddings manejable, pero podés codificar prácticamente cualquier string.

In [ ]:
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding('gpt2')
demo = 'Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.'
integers = gpt2_tokenizer.encode(demo, allowed_special={'<|endoftext|>'})
print('IDs de tokens:', integers)
print('Round-trip:', gpt2_tokenizer.decode(integers))

## Por qué el muestreo con ventana deslizante (max_length + stride)

Los LLMs estilo GPT se entrenan con **predicción del siguiente token**. Eso significa que cada muestra de entrenamiento es un chunk de tokens (inputs) y los targets son el mismo chunk desplazado 1 posición.

- `max_length` ≈ longitud de contexto que el modelo ve durante entrenamiento.
- `stride` controla el solapamiento entre muestras consecutivas.

El solapamiento es útil porque expone al modelo a **más transiciones de borde** (tokens cerca de los extremos). Pero demasiado solapamiento aumenta redundancia y hace el entrenamiento menos eficiente.

In [ ]:
enc_text = gpt2_tokenizer.encode(raw_text)
print('Total de tokens GPT-2 en el corpus:', len(enc_text))

enc_sample = enc_text[50:]
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print('x:', x)
print('y:', y)
print('x decodificado:', gpt2_tokenizer.decode(x))
print('y decodificado:', gpt2_tokenizer.decode(y))

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})
        if len(token_ids) <= max_length:
            raise ValueError('El texto es demasiado corto para el max_length dado')

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tok = tiktoken.get_encoding('gpt2')
    dataset = GPTDatasetV1(txt, tok, max_length=max_length, stride=stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

# Chequeo rápido
dl = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
it = iter(dl)
inputs, targets = next(it)
print('Inputs:', inputs)
print('Targets:', targets)

## Experimento: cambiar `max_length` y `stride`

Objetivo: medir cuántas muestras de entrenamiento obtenemos cuando cambiamos el tamaño del chunk (`max_length`) y el solapamiento (`stride`).

Deberías ver: `stride` más chico → **más muestras** (más solapamiento). `max_length` más grande → **menos muestras** (cada muestra consume más tokens).

In [ ]:
def num_samples(num_tokens: int, max_length: int, stride: int) -> int:
    # Coincide con: len(range(0, num_tokens - max_length, stride))
    stop = num_tokens - max_length
    if stop <= 0:
        return 0
    return ((stop - 1) // stride) + 1

N = len(enc_text)
print('Total de tokens N =', N)

configs = [
    (16, 16),   # sin solapamiento
    (16, 8),    # 50% solapamiento
    (32, 32),
    (32, 16),
    (64, 64),
    (64, 32),
]

tok = tiktoken.get_encoding('gpt2')
for max_length, stride in configs:
    ds = GPTDatasetV1(raw_text, tok, max_length=max_length, stride=stride)
    print(f'max_length={max_length:>3} stride={stride:>3}  -> samples={len(ds):>4}  (fórmula={num_samples(N, max_length, stride):>4})')

### Por qué el solapamiento sirve

Si usamos `stride = max_length`, cada token aparece en **exactamente una** ventana de entrenamiento (salvo efectos de borde). Eso puede sub-entrenar comportamientos de borde: tokens cerca del inicio/fin de cada chunk se ven menos veces como contexto *interior*.

Con solapamiento (stride menor), el mismo token participa en múltiples ventanas y aparece en diferentes posiciones relativas. Esto da supervisión más rica sobre transiciones en los límites (clave cuando luego entrenás con contextos largos). El trade-off es la redundancia: demasiado solapamiento implica más tokens repetidos por época y entrenamiento menos eficiente.

## ¿Por qué los embeddings codifican significado? (y relación con conceptos de redes neuronales)

Los embeddings son **parámetros aprendidos**: cada ID de token indexa una fila en una matriz de embeddings $E \in \mathbb{R}^{V \times d}$. Durante el entrenamiento, los gradientes actualizan estos vectores de manera que tokens que aparecen en **contextos predictivos similares** reciben actualizaciones en direcciones similares.

Por eso codifican “significado”: no porque el significado esté hardcodeado, sino porque el objetivo de entrenamiento obliga a que esos vectores se vuelvan **features útiles** para predecir el siguiente token. Tokens con roles similares (sintaxis/semántica) tienden a quedar más cerca en el espacio porque ayudan a hacer predicciones parecidas.

Relación con redes neuronales: un lookup de embedding es equivalente a **one-hot + una capa lineal**. Si $x$ es un vector one-hot, entonces $xE$ selecciona una fila de $E$. O sea: la capa de embeddings es una capa NN estándar (pesos entrenables), solo que implementada eficientemente como indexación.

In [ ]:
# Ejemplo juguete de embeddings de tokens
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print('Forma de la matriz de pesos:', embedding_layer.weight.shape)
print('Embedding para id=3:', embedding_layer(torch.tensor([3])))
print('Embeddings para [2,3,5,1]:
', embedding_layer(input_ids))

In [ ]:
# Combinar embeddings de token + embeddings posicionales (GPT-2: posiciones absolutas)
max_length = 4
dl = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
inputs, targets = next(iter(dl))

vocab_size = 50257
emb_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, emb_dim)
pos_embedding_layer = torch.nn.Embedding(max_length, emb_dim)

token_embeddings = token_embedding_layer(inputs)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
input_embeddings = token_embeddings + pos_embeddings

print('inputs shape:', inputs.shape)
print('token embeddings shape:', token_embeddings.shape)
print('pos embeddings shape:', pos_embeddings.shape)
print('input embeddings final shape:', input_embeddings.shape)

## Cierre

Ya tenemos el pipeline completo que espera un loop de entrenamiento de un LLM: IDs de tokens, muestras de entrenamiento (inputs/targets) vía ventana deslizante, y representaciones por embeddings (token + posición). En capítulos posteriores, estos embeddings pasan a ser la entrada de atención (attention) y capas MLP.